# eye_cnn.ipynb — CNN 눈 뜸/감음 학습 (B, P2, 2일 제한)

- 데이터: **MRL Eye Dataset** (Kaggle `akashshingha850/mrl-eye-dataset`) — 이 데이터만 사용
- 수업 **딥러닝 14차시** 전이학습 노트북을 복사해서 시작
- 받으면 먼저 **파일 이름에 사람 번호**가 있는지 확인 → **사람 단위로** train / test 분할
- 모두 흑백, 64×64, 밝기 · 대비 · 흐림 증강 (8차시)
- 결과: `eye_cnn.pt` → 서버 `ml_models/` (50MB 넘으면 Google Drive)
- 3주차 수요일: 팀 사진으로 ML Kit · EAR 과 비교 → 채택 결정 (KPI ①)
- `kaggle.json` 은 절대 저장소에 올리지 않음. 실행 결과(출력)는 지우고 올림


## 1. MRL 데이터 확인 (학습 전에 먼저)

팀장님 주의 사항 기준으로 확인할 것 — 결과는 맨 아래 표에 적어 노션에 올린다.

| # | 확인할 것 | 셀 |
|---|---|---|
| 1 | 장수 84,898 · 이름 규칙 · 중복 | 1-3 |
| 2 | 폴더 라벨(awake/sleepy) = 파일 이름 눈 상태(1/0) — **전부** | 1-4 |
| 3 | Kaggle 기존 train/val/test 에 같은 사람이 섞였는지 — **전부** | 1-5 |
| 4 | 피험자별 장수 · 한 클래스만 있는 사람 · 안경 · 센서 | 1-6 |
| 5 | 이미지 크기 · 흑백 여부 · 가로세로 비 | 1-7 |
| 6 | 눈으로 보기 — 뜸/감음 · 센서 · 안경 · 반사 · 가장 작은/큰 사진 | 1-8 |
| 7 | **눈이 얼마나 꽉 차게 잘렸나** → 서버에서 자를 여유(margin) 정하기 | 1-9 |
| 8 | 64×64 흑백으로 줄였을 때 밝기 평균 · 표준편차 → 정규화 값 후보 | 1-10 |
| 9 | 사람 단위 분할 + 감음/뜸 비율 균형 | 1-11 |

- 🔴 `kaggle.json` 은 **구글 드라이브에만**. 저장소 · 노트북 출력 · 채팅에 올리지 않는다
- 🔴 GitHub 에 올리기 전: Colab → 수정 → **모든 출력 지우기** (MRL 사진이 출력에 남으면 안 됨)
- 결과 그림 · 분할표는 `OUT_DIR`(드라이브)에 저장된다

### 1-1. 드라이브 연결 + Kaggle 키

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, shutil
KAGGLE_JSON = '/content/drive/MyDrive/keys/kaggle.json'        # ← 본인 kaggle.json 위치
OUT_DIR = '/content/drive/MyDrive/sleep_detector/mrl_check'    # ← 결과 그림 · 분할표 저장 폴더
os.makedirs(OUT_DIR, exist_ok=True)

os.makedirs(os.path.expanduser('~/.kaggle'), exist_ok=True)
shutil.copy(KAGGLE_JSON, os.path.expanduser('~/.kaggle/kaggle.json'))
os.chmod(os.path.expanduser('~/.kaggle/kaggle.json'), 0o600)
print('키 준비 완료 (내용은 출력하지 않음)')

### 1-2. 내려받기 (Colab 로컬 디스크, 약 330MB — 드라이브에 풀지 않는다)

In [ ]:
%%bash
if [ ! -d /content/mrl ]; then
  pip -q install kaggle
  kaggle datasets download -d akashshingha850/mrl-eye-dataset -p /content/mrl --unzip -q
fi
echo "폴더:"; find /content/mrl -maxdepth 3 -type d | head -20
echo "PNG 수:"; find /content/mrl -name '*.png' | wc -l

### 1-3. 파일 이름 읽기 — 장수 · 이름 규칙 · 중복

In [ ]:
import glob, re, collections
from pathlib import Path
import pandas as pd

# s0001_00001_0_0_0_0_0_01 = 피험자_번호_성별_안경_눈상태_반사_조명_센서
NAME = re.compile(r'^(s\d{4})_(\d+)_([01])_([01])_([01])_([012])_([01])_(\d{2})$')
paths = sorted(glob.glob('/content/mrl/**/*.png', recursive=True))
rows, bad = [], []
for p in paths:
    stem = os.path.basename(p)[:-4]
    m = NAME.match(stem)
    if not m:
        bad.append(p)
        continue
    parts = set(Path(p).parts)
    rows.append(dict(path=p, name=stem, subject=m[1], img_no=int(m[2]), gender=int(m[3]), glasses=int(m[4]),
                     eye=int(m[5]), reflection=int(m[6]), light=int(m[7]), sensor=m[8],
                     kaggle_split=next((s for s in ('train', 'val', 'test') if s in parts), '?'),
                     folder=next((s for s in ('awake', 'sleepy') if s in parts), '?')))
df = pd.DataFrame(rows)

print('PNG 전체      :', len(paths))
print('이름 규칙 통과 :', len(df), '/ 불통과:', len(bad), bad[:5])
print('84,898 장인가  :', len(df) == 84898)
print('같은 이름 중복 :', int(df['name'].duplicated().sum()))
print('피험자 수      :', df['subject'].nunique())

### 1-4. 폴더 라벨 = 파일 이름 눈 상태? (전부)

- 기대: `sleepy` = 0(감음), `awake` = 1(뜸), 불일치 0장

In [ ]:
df['folder_eye'] = df['folder'].map({'sleepy': 0, 'awake': 1})
mismatch = df[df['folder_eye'] != df['eye']]
print('폴더 라벨 ≠ 파일 이름 눈 상태:', len(mismatch), '장')
display(pd.crosstab(df['folder'], df['eye'], margins=True))
display(mismatch.head(10)[['name', 'kaggle_split', 'folder', 'eye']])

### 1-5. Kaggle 기존 분할에 같은 사람이 섞였나? (전부)

- 섞였으면 기존 train/val/test 는 쓰지 않고 1-11 의 사람 단위 분할을 쓴다

In [ ]:
subj_split = df.groupby('subject')['kaggle_split'].agg(lambda s: '+'.join(sorted(set(s))))
print(subj_split.value_counts().to_string())
print('\n여러 split 에 걸친 피험자:', int((subj_split.str.contains('\\+')).sum()), '/', len(subj_split))
display(pd.crosstab(df['subject'], df['kaggle_split']).head(10))

### 1-6. 피험자별 장수 · 클래스 · 안경 · 센서

In [ ]:
per = (df.pivot_table(index='subject', columns='eye', values='name', aggfunc='count', fill_value=0)
         .rename(columns={0: 'closed', 1: 'open'}))
per['total'] = per['closed'] + per['open']
per['closed_ratio'] = (per['closed'] / per['total']).round(2)
per['glasses_ratio'] = df.groupby('subject')['glasses'].mean().round(2)
per['gender'] = df.groupby('subject')['gender'].agg(lambda s: s.mode()[0])
per['sensors'] = df.groupby('subject')['sensor'].agg(lambda s: ','.join(sorted(set(s))))
display(per.sort_values('total'))

print('한 클래스만 있는 피험자:', list(per[(per['closed'] == 0) | (per['open'] == 0)].index))
print('장수 최소 / 최대:', per['total'].idxmin(), per['total'].min(), '/', per['total'].idxmax(), per['total'].max())
for col in ['gender', 'glasses', 'eye', 'reflection', 'light', 'sensor']:
    print(f'{col:10s}', df[col].value_counts().sort_index().to_dict())

### 1-7. 이미지 크기 · 흑백 여부 (전부, 머리 부분만 읽어서 빠름)

In [ ]:
from PIL import Image
import matplotlib.pyplot as plt

sizes, modes = [], collections.Counter()
for p in df['path']:
    with Image.open(p) as im:
        sizes.append(im.size)
        modes[im.mode] += 1
df['w'] = [s[0] for s in sizes]
df['h'] = [s[1] for s in sizes]
df['aspect'] = (df['w'] / df['h']).round(2)

print('이미지 모드 (L = 흑백):', dict(modes))
display(df[['w', 'h', 'aspect']].describe().round(1))
print('센서별 중간값'); display(df.groupby('sensor')[['w', 'h', 'aspect']].median())
print('뜸/감음별 중간값'); display(df.groupby('eye')[['w', 'h', 'aspect']].median())

fig, ax = plt.subplots(1, 2, figsize=(10, 3))
ax[0].hist(df['w'], bins=60); ax[0].set_title('width (px)')
ax[1].hist(df['aspect'], bins=60); ax[1].set_title('width / height')
plt.tight_layout(); plt.savefig(f'{OUT_DIR}/size_hist.png', dpi=100); plt.show()

### 1-8. 눈으로 보기

- 제목: `피험자 e눈상태 g안경 / 가로x세로 s센서` — 라벨이 틀려 보이는 사진, 반쯤 감은 사진이 있는지 본다

In [ ]:
def grid(sub, title, fname, n=24, cols=8, seed=0):
    sub = sub.sample(min(n, len(sub)), random_state=seed)
    nrows = (len(sub) + cols - 1) // cols
    fig, axes = plt.subplots(nrows, cols, figsize=(cols * 1.6, nrows * 1.9))
    for ax in axes.flat:
        ax.axis('off')
    for ax, (_, r) in zip(axes.flat, sub.iterrows()):
        ax.imshow(Image.open(r.path), cmap='gray', vmin=0, vmax=255)
        ax.set_title(f'{r.subject} e{r.eye} g{r.glasses}\n{r.w}x{r.h} s{r.sensor}', fontsize=7)
    fig.suptitle(title)
    plt.tight_layout(); plt.savefig(f'{OUT_DIR}/{fname}', dpi=100); plt.show()

grid(df[df.eye == 1], 'open (eye=1)', 'grid_open.png')
grid(df[df.eye == 0], 'closed (eye=0)', 'grid_closed.png')
for s in sorted(df['sensor'].unique()):
    grid(df[df.sensor == s], f'sensor {s}', f'grid_sensor{s}.png', n=16)
grid(df[df.glasses == 1], 'glasses', 'grid_glasses.png', n=16)
grid(df[df.reflection == 2], 'big reflection', 'grid_reflection.png', n=16)
grid(df[df.light == 0], 'bad light', 'grid_badlight.png', n=16)
grid(df.nsmallest(16, 'w'), 'smallest', 'grid_smallest.png', n=16)
grid(df.nlargest(16, 'w'), 'largest', 'grid_largest.png', n=16)

### 1-9. 눈이 얼마나 꽉 차게 잘렸나 (서버 자르기 여유 정하기)

- 빨간 선 = 사진 가로 · 세로의 10% 간격. 뜬 눈 16장을 보고 아래를 적는다
- 이 값으로 서버의 **MediaPipe 눈 점 → 자르기 범위**를 정한다 (학습과 서버가 같은 모양이어야 함)

In [ ]:
sub = df[df.eye == 1].sample(16, random_state=1)
fig, axes = plt.subplots(2, 8, figsize=(16, 4.6))
for ax, (_, r) in zip(axes.flat, sub.iterrows()):
    ax.imshow(Image.open(r.path).convert('L').resize((130, 130)), cmap='gray', vmin=0, vmax=255)
    for t in range(0, 131, 13):
        ax.axvline(t, color='r', lw=0.8, alpha=0.6); ax.axhline(t, color='r', lw=0.8, alpha=0.6)
    ax.set_xlim(0, 129); ax.set_ylim(129, 0)  # 선 때문에 그림 밖에 흰 여백이 생기지 않게
    ax.set_xticks([]); ax.set_yticks([]); ax.set_title(f'{r.subject} {r.w}x{r.h}', fontsize=8)
plt.tight_layout(); plt.savefig(f'{OUT_DIR}/crop_tightness.png', dpi=100); plt.show()

**1-9 결과 (10.14 실행, 1-8 · 1-9 그림을 눈으로 본 값 — 1-10b 평균 사진으로 다시 확인)**

- 사진은 **전부 정사각형** (가로세로 비 1.0, 84,898장 모두)
- 눈 가로 폭(눈꼬리~눈꼬리): 사진 가로의 약 **55~70%** (센서 02 의 s0031 은 약 35~40% 로 눈이 작게 찍힘)
- 눈은 가로 가운데, 세로는 가운데보다 **약간 아래** (눈 중심이 위에서 약 50~55%)
- 눈썹: **들어 있는 사진이 많음** (위쪽 20~30% 안), 사진마다 다름 → 자르는 범위가 사진마다 조금씩 다르다
- 라벨이 틀려 보이는 사진: 이번 표본에서는 못 찾음. 반쯤 감은 눈은 '뜸'(e1) 쪽에 일부 있음 (예: s0014 · s0019 의 가늘게 뜬 눈)

### 1-10. 64×64 흑백으로 줄였을 때 밝기 (정규화 값 후보)

- 피험자 · 눈 상태마다 최대 100장씩 표본
- ⚠️ 줄이는 방법(보간)도 서버와 같아야 한다 — 여기서는 PIL BILINEAR, 서버가 cv2 를 쓰면 `cv2.INTER_AREA` 등으로 맞춘다

In [ ]:
import numpy as np
SIZE = 64  # 서버 config.CNN_INPUT_SIZE

samp = df.sample(frac=1, random_state=0).groupby(['subject', 'eye']).head(100)  # 피험자 · 눈 상태마다 최대 100장
arr = np.stack([np.asarray(Image.open(p).convert('L').resize((SIZE, SIZE), Image.BILINEAR), dtype=np.float32) / 255
                for p in samp['path']])
print(f'표본 {len(arr)}장 / 평균 {arr.mean():.4f} / 표준편차 {arr.std():.4f}  ← 정규화 값 후보')
samp = samp.assign(mean=arr.mean(axis=(1, 2)))
print('센서별 밝기'); display(samp.groupby('sensor')['mean'].describe().round(3))
print('뜸/감음별 밝기'); display(samp.groupby('eye')['mean'].describe().round(3))

### 1-10b. 평균 눈 사진 — 눈이 사진의 어디에, 얼마나 크게 있나 (1-9 를 숫자로 확인)

- 1-10 의 64×64 표본을 뜸 / 감음으로 나눠 평균. 어두운 띠 = 눈이 있는 자리
- 자르기 범위는 **뜬 눈 평균**으로 잰다. 사진 가장자리는 원래 어두워서(비네팅) 가장자리 15%는 빼고 찾는다
- 눈 폭 = 눈 높이의 가로 띠에서 '그 열의 평균보다 어두운 정도'가 가장 어두운 곳의 절반 이상인 구간

In [ ]:
def inner_argmin(p, lo=0.15, hi=0.85):
    a, b = int(len(p) * lo), int(len(p) * hi)
    return a + int(np.argmin(p[a:b]))

m_open = arr[(samp['eye'] == 1).to_numpy()].mean(axis=0)
m_closed = arr[(samp['eye'] == 0).to_numpy()].mean(axis=0)

# 눈 중심 = 가장자리 15% 를 뺀 안쪽에서 가장 어두운 점 (5×5 평균으로 부드럽게) — 가로줄 평균은 눈썹 띠에 끌려감
pad = np.pad(m_open, 2, mode='edge')
smooth = sum(pad[dy:dy + SIZE, dx:dx + SIZE] for dy in range(5) for dx in range(5)) / 25
a, b = int(SIZE * 0.15), int(SIZE * 0.85)
cy, cx = np.unravel_index(np.argmin(smooth[a:b, a:b]), (b - a, b - a))
cy, cx = int(cy) + a, int(cx) + a
dark = m_open[max(cy - 3, 0):cy + 4].mean(axis=0) - m_open.mean(axis=0)  # 눈 띠가 그 열 평균보다 얼마나 어두운가
def span(frac):
    level = dark[cx] * frac  # 가장 어두운 깊이의 frac 만큼 어두운 곳까지
    left = right = cx
    while left > 0 and dark[left - 1] < level:
        left -= 1
    while right < SIZE - 1 and dark[right + 1] < level:
        right += 1
    return left, right

left, right = span(0.5)
l20, r20 = span(0.2)
eye_w50, eye_w20 = (right - left + 1) / SIZE, (r20 - l20 + 1) / SIZE

print(f'뜬 눈 중심: 위에서 {cy / SIZE:.0%}, 왼쪽에서 {cx / SIZE:.0%}')
print(f'눈 폭 — 절반 깊이: 사진 가로의 {eye_w50:.0%} / 20% 깊이: {eye_w20:.0%}')
print('  (평균 사진은 흐려서 눈꼬리가 옅다 → MediaPipe 의 눈꼬리~눈꼬리 폭에는 20% 깊이 값이 더 가깝다. 1-9 눈 확인값 55~70% 와 비교)')
print(f'  → 자르기 한 변 S ≈ 눈 폭 W ÷ {eye_w20:.2f} = {1 / eye_w20:.2f}W  (20% 깊이 기준)')
print(f'자르기 중심 = 눈 중심보다 {cy / SIZE - 0.5:+.2f}S 아래에 눈이 오도록 (음수면 위)')
brow = inner_argmin(m_closed.mean(axis=1), 0.10, 0.45)
print(f'감은 눈 평균의 눈썹 띠: 위에서 {brow / SIZE:.0%}')

fig, ax = plt.subplots(1, 3, figsize=(12, 3.4))
for a, m, name in ((ax[0], m_open, 'open mean'), (ax[1], m_closed, 'closed mean')):
    a.imshow(m, cmap='gray')
    for t in np.linspace(0, SIZE - 1, 11):
        a.axvline(t, color='r', lw=0.8, alpha=0.6); a.axhline(t, color='r', lw=0.8, alpha=0.6)
    a.set_xlim(0, SIZE - 1); a.set_ylim(SIZE - 1, 0); a.set_title(name)
ax[0].axvspan(left, right, ymin=0.45, ymax=0.55, color='y', alpha=0.4)
ax[2].plot(np.arange(SIZE) / SIZE, m_open.mean(axis=1), label='open rows (top→bottom)')
ax[2].plot(np.arange(SIZE) / SIZE, dark, '--', label='open eye band - col mean (left→right)')
ax[2].plot(np.arange(SIZE) / SIZE, m_closed.mean(axis=1), label='closed rows (top→bottom)')
ax[2].axhline(dark[cx] * 0.5, color='gray', lw=0.6); ax[2].axhline(dark[cx] * 0.2, color='gray', lw=0.6, ls=':'); ax[2].set_xlabel('position (0 = top/left, 1 = bottom/right)'); ax[2].legend(fontsize=7)
plt.tight_layout(); plt.savefig(f'{OUT_DIR}/mean_eye.png', dpi=100); plt.show()

### 1-11. 사람 단위 분할 (70 / 15 / 15) + 감음/뜸 비율 균형

1. 장수가 많은 사람부터 목표 비율에 가장 모자란 쪽에 배정 (D:\scratch 의 처음 분할안과 같은 방식)
2. 피험자 두 명을 맞바꿔서 **장수 비율과 감음 비율이 목표에 더 가까워지면** 바꾼다 — 더 나아지지 않을 때까지 (같은 데이터면 항상 같은 결과)

In [ ]:
import itertools, json, datetime

TARGET = {'train': 0.70, 'val': 0.15, 'test': 0.15}
total = int(per['total'].sum())
overall_closed = per['closed'].sum() / total

split, cnt = {}, collections.Counter()
for s in per.sort_values('total', ascending=False).index:
    k = min(TARGET, key=lambda k: cnt[k] / total - TARGET[k])
    split[s] = k
    cnt[k] += per.loc[s, 'total']

def stats(split):
    out = {}
    for k in TARGET:
        ss = sorted(s for s in split if split[s] == k)
        n, c = int(per.loc[ss, 'total'].sum()), int(per.loc[ss, 'closed'].sum())
        out[k] = dict(subjects=ss, n_subjects=len(ss), images=n, ratio=round(n / total, 3),
                      closed=c, open=n - c, closed_ratio=round(c / n, 3))
    return out

def cost(split):
    st = stats(split)
    return sum(2 * abs(st[k]['ratio'] - TARGET[k]) + abs(st[k]['closed_ratio'] - overall_closed) for k in TARGET)

def show(title, st):
    print(title)
    for k, v in st.items():
        print(f"  {k:5s} {v['n_subjects']:2d}명 {v['images']:6d}장 ({v['ratio']:.1%})  감음 {v['closed_ratio']:.1%}")

show('처음 배정', stats(split))
while True:
    best_c, best_pair = cost(split), None
    for a, b in itertools.combinations(sorted(split), 2):
        if split[a] == split[b]:
            continue
        split[a], split[b] = split[b], split[a]
        c = cost(split)
        split[a], split[b] = split[b], split[a]
        if c < best_c - 1e-9:
            best_c, best_pair = c, (a, b)
    if best_pair is None:
        break
    a, b = best_pair
    split[a], split[b] = split[b], split[a]

final = stats(split)
show(f'\n균형 조정 후 (전체 감음 비율 {overall_closed:.1%})', final)
for k, v in final.items():
    print(f"{k}: {', '.join(v['subjects'])}")
one_class = list(per[(per['closed'] == 0) | (per['open'] == 0)].index)
print('\n한 클래스만 있는 피험자의 배정:', {s: split[s] for s in one_class})

json.dump({'made': str(datetime.date.today()), 'method': 'greedy + swap balance', 'split': split, 'summary': final},
          open(f'{OUT_DIR}/mrl_person_split.json', 'w', encoding='utf-8'), ensure_ascii=False, indent=1)
df['split'] = df['subject'].map(split)
print('\n저장:', f'{OUT_DIR}/mrl_person_split.json')

### 1-12. 결과 요약 (노션에 붙여 넣기)

In [ ]:
summary = {
    '장수': len(df), '84898과 같음': len(df) == 84898, '이름 불통과': len(bad), '이름 중복': int(df['name'].duplicated().sum()),
    '피험자': df['subject'].nunique(),
    '폴더 라벨 불일치': len(mismatch),
    'Kaggle 분할에서 여러 split 에 걸친 피험자': int(subj_split.str.contains('\\+').sum()),
    '한 클래스만 있는 피험자': one_class,
    '이미지 모드': dict(modes),
    '크기 중간값 (w, h)': (int(df['w'].median()), int(df['h'].median())),
    '크기 범위 w': (int(df['w'].min()), int(df['w'].max())),
    '64x64 밝기 평균 / 표준편차': (round(float(arr.mean()), 4), round(float(arr.std()), 4)),
    '분할': {k: f"{v['n_subjects']}명 {v['images']}장 감음 {v['closed_ratio']:.1%}" for k, v in final.items()},
}
for k, v in summary.items():
    print(f'- {k}: {v}')

### 1-13. 확인 결과 정리 (10.14 실행)

| 확인 | 결과 | 판정 |
|---|---|---|
| 장수 · 이름 규칙 · 중복 | 84,898장, 이름 규칙 불통과 0, 중복 0, 피험자 37명 | ✅ |
| 폴더 라벨 = 파일 이름 눈 상태 | 불일치 **0장** (sleepy = 0 감음 41,946 / awake = 1 뜸 42,952) | ✅ |
| Kaggle 기존 분할 | **37명 전원**이 train · val · test 에 다 들어 있음 → 이미지 단위 무작위 분할 | ❌ 쓰지 않음 → 1-11 분할 사용 |
| 이미지 | 전부 흑백(L), 전부 정사각형, 52~299px (중간값 87) | ✅ |
| 센서 | 01 이 83% (70,142장). 센서마다 크기(01: 84 · 02: 126 · 03: 73px) · 밝기가 다름 | ⚠️ |
| **피험자별 감음 비율** | 0.02 ~ 1.00 으로 **극단적** (s0007 98% 감음, s0035 3% 감음) | ⚠️ 사람 단위 분할 + 비율 균형이 꼭 필요 |
| 한 클래스만 있는 피험자 | s0004 · s0008 (감음만) → 둘 다 test | ⚠️ test 의 감음 재현율이 두 사람에 영향받음 |
| 뜸 / 감음 사진 크기 | 감음 중간값 91px, 뜸 82px — 크기가 라벨과 관련 | ⚠️ 64×64 로 줄이면 크기 자체는 사라지지만 눈이 차지하는 비율 차이는 남을 수 있음 |
| 64×64 밝기 | 평균 0.3426, 표준편차 0.1024 | 정규화 후보 |
| 분할 (1-11) | train 16명 59,560 / val 10명 12,724 / test 11명 12,614, **세 구간 모두 감음 49.4%** | ✅ 이 분할로 학습 |

**공통 전처리 함수 제안 (A · 팀장님과 정하기 — 노트북과 `face_service.py` 가 같은 코드)**

- 서버 (폰 컬러 얼굴 사진): MediaPipe 눈꼬리 점(33 · 133 / 362 · 263)으로 눈 가로 폭 W, 두 점의 가운데 = 눈 중심
  → **한 변 S = 1.5W 정사각형**, 중심은 **눈 중심보다 0.03S 위** (1-10b 확정값 — 눈이 위에서 53% 에 오도록)
  → 흑백 → 64×64 (보간 방법 통일, 예: `cv2.INTER_AREA`) → 정규화
- MRL (이미 눈만 잘린 사진): 흑백 → 64×64 → 정규화 (자르기 없음)
- 정규화: ① 데이터 전체 값 `(x − 0.3426) ÷ 0.1024` ② **사진마다** `(x − 평균) ÷ 표준편차`
  → MRL 은 적외선, 폰은 컬러라 밝기 차이가 크므로 ② 를 기본으로 추천. 최종은 팀 사진(KPI ①)으로 비교
- MRL 의 자르기 범위가 사진마다 달라서 학습 증강에 **크기 ±15% · 위치 ±5% 흔들기**를 추가 (기존: 밝기 · 대비 · 흐림 · ±10° 회전 · 좌우 반전)
- **자르기 범위 확정 (1-10b, 10.14 세 번째 실행)**: 뜬 눈 평균의 눈 중심 위에서 53% · 왼쪽에서 50%,
  눈 폭 20% 깊이 66% (절반 깊이 47%) → S ≈ 1.52W. 1-9 눈 확인값 55~70% (≈ 1.4~1.8W) 와 맞음 → **S = 1.5W** 로 정함.
  감은 눈 평균의 눈썹 띠는 위에서 23%

**남은 일**

- MRL 출처 표기 (발표 자료): 원 논문 인용 문구 확인
- 🔴 저장소에 올리기 전 **모든 출력 지우기**